# Apertus empirical cost study

Executable harness for Zepto vs CUDA twin measurements. Plan: [`docs/plans/empirical-cost-study-plan.md`](../docs/plans/empirical-cost-study-plan.md). Per-run methodology: `output_dir/methodology.md`.

## 1. Setup

In [ ]:
from __future__ import annotations

import sys
from datetime import datetime
from pathlib import Path

REPO = Path.cwd().resolve()
if (REPO / "src" / "zepto").is_dir():
    sys.path.insert(0, str(REPO / "src"))
elif (REPO.parent / "src" / "zepto").is_dir():
    sys.path.insert(0, str(REPO.parent / "src"))
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA PyTorch is required for the empirical smoke milestone.")

print(f"torch {torch.__version__}")
print(f"device: {torch.cuda.get_device_name(0)}")

## 2. Run config (edit here)

In [ ]:
from zepto.empirical.runner import RunConfig, run_study
from zepto.empirical.sampler import ArchitectureRanges, IntRange, SamplerConfig
from tests.integration.apertus.shared import GOLDEN

RUN_SLUG = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR = REPO / "notebooks" / "artifacts" / "apertus_empirical" / RUN_SLUG

SAMPLER = SamplerConfig(
    seq_len=IntRange(8, 16),
    batch_size=IntRange(1, 2),
    architecture=ArchitectureRanges(
        head_dim=IntRange(GOLDEN["head_dim"], GOLDEN["head_dim"]),
        intermediate_size=IntRange(GOLDEN["intermediate_size"], GOLDEN["intermediate_size"]),
        num_heads=IntRange(GOLDEN["num_heads"], GOLDEN["num_heads"]),
        num_kv_heads=IntRange(GOLDEN["num_kv_heads"], GOLDEN["num_kv_heads"]),
        num_layers=IntRange(GOLDEN["num_layers"], GOLDEN["num_layers"]),
        vocab_size=IntRange(GOLDEN["vocab_size"], GOLDEN["vocab_size"]),
    ),
    architecture_mins={
        "hidden_size": GOLDEN["hidden_size"],
        "intermediate_size": GOLDEN["intermediate_size"],
        "vocab_size": GOLDEN["vocab_size"],
        "num_layers": GOLDEN["num_layers"],
    },
)

CONFIG = RunConfig(
    output_dir=OUTPUT_DIR,
    master_seed=0,
    model_id="apertus",
    n_configs=1,
    m_draws_per_config=1,
    training_steps_per_draw=3,
    sampler=SAMPLER,
)

print(f"output_dir={OUTPUT_DIR}")

## 3. Execute

In [ ]:
import logging

logging.basicConfig(level=logging.INFO)
run_study(CONFIG)

## 4. Preview

In [ ]:
import pandas as pd

eval_path = OUTPUT_DIR / "evaluation.csv"
df = pd.read_csv(eval_path)
display(df.head())
print(df.groupby(["phase", "step"]).size())

## 5. Figures

In [ ]:
import matplotlib.pyplot as plt

df["flop_rel_err"] = df["target_flop"] / df["y_flop"] - 1
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, phase in zip(axes, ["inference", "training"]):
    sub = df[df["phase"] == phase]
    ax.scatter(sub["seq_len"], sub["flop_rel_err"], c=sub["batch_size"].map({1: "C0", 2: "C1"}))
    ax.set_title(phase)
    ax.set_xlabel("seq_len")
    ax.set_ylabel("target_flop/y_flop - 1")
    ax.axhline(0, color="k", lw=0.5)
plt.tight_layout()
plt.show()